In [ ]:
# ============================================================
# PHARMA EVIDENCE API — Day 1 Harvesting Script
# Pulls KRAS G12C inhibitor trials in NSCLC from ClinicalTrials.gov
# ============================================================

# STEP 1: Install/import the libraries we need
# 'requests' lets Python make web/API calls (like a browser, but for code)
# 'pandas' is the table/spreadsheet library you're learning
import requests
import pandas as pd

# STEP 2: Define the API endpoint and our search parameters
# NOTE: The real ClinicalTrials.gov v2 API uses 'params' as a dictionary,
# not one giant URL string. Python builds the URL for us — this avoids
# typos and makes it easy to change search terms later.
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.intr": "adagrasib OR sotorasib OR divarasib",  # Intervention (drug) search
    "query.cond": "NSCLC",                                  # Condition/disease search
    "filter.overallStatus": "COMPLETED",                     # Only completed trials
    "pageSize": 100,                                         # Max results per page (max allowed: 1000)
    "format": "json"                                         # We want JSON back, not CSV
}

# STEP 3: Make the actual API call
# requests.get() sends the request; the API sends back a response object
print("Calling ClinicalTrials.gov API...")
response = requests.get(BASE_URL, params=params)

# STEP 4: Check if the call worked before doing anything else
# Status code 200 = success. Anything else = something went wrong.
if response.status_code == 200:
    print(f"✅ Success! Status code: {response.status_code}")
    data = response.json()  # Convert the raw response text into a Python dictionary
    print(f"Total matching studies found: {data.get('totalCount', 'unknown')}")
else:
    print(f"❌ Something went wrong. Status code: {response.status_code}")
    print(response.text)  # Print the error message the API sent back

# STEP 5: Pull out the list of individual studies
# The JSON structure is nested: data -> "studies" -> [list of study records]
studies = data.get("studies", [])
print(f"Studies returned in this page: {len(studies)}")

# STEP 6: Extract the fields we actually care about from each study
# IMPORTANT: Unlike the flat field names in the original request idea
# (NCTId, BriefTitle, etc.), the real API nests everything inside
# "protocolSection" -> various "modules". We have to dig into that
# structure to pull each value out safely.
rows = []

for study in studies:
    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    design = protocol.get("designModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    outcomes = protocol.get("outcomesModule", {})
    interventions = protocol.get("armsInterventionsModule", {})
    has_results = study.get("hasResults", False)

    # .get() with a default value (like {} or "Unknown") prevents the script
    # from crashing if a field is missing for a particular trial — this
    # happens a LOT with real-world clinical trial data.
    nct_id = identification.get("nctId", "Unknown")
    title = identification.get("briefTitle", "Unknown")
    phase_list = design.get("phases", [])
    phase = ", ".join(phase_list) if phase_list else "Not specified"
    enrollment = design.get("enrollmentInfo", {}).get("count", None)
    lead_sponsor = sponsor.get("leadSponsor", {}).get("name", "Unknown")

    # Intervention names: this is a list of dicts, so we pull out just the "name" field from each
    intervention_names = [i.get("name", "") for i in interventions.get("interventions", [])]
    intervention_str = "; ".join(intervention_names)

    # Primary outcomes: also a list of dicts — grab just the measure description
    primary_outcomes = outcomes.get("primaryOutcomes", [])
    primary_outcome_str = "; ".join([po.get("measure", "") for po in primary_outcomes])

    rows.append({
        "NCTId": nct_id,
        "BriefTitle": title,
        "Phase": phase,
        "EnrollmentCount": enrollment,
        "LeadSponsor": lead_sponsor,
        "Interventions": intervention_str,
        "PrimaryOutcomes": primary_outcome_str,
        "HasResultsPosted": has_results
    })

# STEP 7: Turn our list of dictionaries into a Pandas DataFrame (a table)
df = pd.DataFrame(rows)

# STEP 8: Quick sanity checks — this is how you "verify it worked"
print("\n--- DataFrame shape (rows, columns) ---")
print(df.shape)

print("\n--- First 5 rows ---")
print(df.head())

print("\n--- Column data types ---")
print(df.dtypes)

# STEP 9 (optional but useful): Save it to a CSV file so you don't
# have to re-call the API every time you want to look at the data
df.to_csv("kras_g12c_nsclc_trials.csv", index=False)
print("\n✅ Saved to kras_g12c_nsclc_trials.csv")

Calling ClinicalTrials.gov API...
✅ Success! Status code: 200
Total matching studies found: unknown
Studies returned in this page: 8

--- DataFrame shape (rows, columns) ---
(8, 8)

--- First 5 rows ---
         NCTId                                         BriefTitle  \
0  NCT04720976  JAB-3312 Based Combination Therapy in Adult Pa...   
1  NCT04330664  Adagrasib in Combination With TNO155 in Patien...   
2  NCT06333678  A Study Comparing Sotorasib With Durvalumab in...   
3  NCT05054725  Combination Study of RMC-4630 and Sotorasib fo...   
4  NCT06314763            Rivaroxaban Sotorasib Interaction Study   

            Phase  EnrollmentCount                             LeadSponsor  \
0  PHASE1, PHASE2               58            Allist Pharmaceuticals, Inc.   
1          PHASE1               86                Mirati Therapeutics Inc.   
2          PHASE2               10  Memorial Sloan Kettering Cancer Center   
3          PHASE2               47              Revolution Medicines, 

In [ ]:
# ============================================================
# TRUST SCORING ENGINE — Statistical Validity Score (v0, placeholder rules)
# Builds on the existing 'df' DataFrame from the harvesting step
# ============================================================

# STEP 1: Define the scoring function.
# In Pandas, when you want to apply custom logic to EVERY row of a
# DataFrame, you write a regular Python function that takes one row
# as input, and then use df.apply() to run it across all rows.
def calculate_trust_score(row):
    """
    Takes a single row (one trial) and returns a Statistical Validity Score (0-100).
    Starts at a perfect 100 and SUBTRACTS points for each red flag found.
    """
    score = 100  # Every trial starts at a perfect score

    # ---- RULE 1: Underpowered Sample Size ----
    # row["EnrollmentCount"] pulls the value out of just THIS row.
    # NOTE: this is a placeholder threshold (n < 30), not the spec's real
    # power-calculation formula (which needs effect size + alpha + power
    # to compute a "minimum detectable n" per trial). Flagging this so it
    # doesn't get mistaken for the final scoring logic later.
    if pd.isna(row["EnrollmentCount"]) or row["EnrollmentCount"] < 30:
        score -= 25

    # ---- RULE 2: P-Hacking Risk / Suspicious Results (placeholder) ----
    # We don't have actual p-values extracted yet (that requires parsing
    # the ResultsSection, which we haven't built). For now this is a
    # placeholder that does nothing until that data exists — it's here
    # so the function's structure matches the spec's 4 rules, but it
    # currently never fires. Flag it as a TODO so it's not forgotten.
    p_hacking_flag = False  # TODO: replace with real p-value check later
    if p_hacking_flag:
        score -= 10

    # ---- RULE 3: Missing Results ----
    # If HasResultsPosted is False, an AI agent has nothing to verify
    # claims against — heavily penalize this.
    if row["HasResultsPosted"] == False:
        score -= 50

    # Never let the score go below 0 (a trial can't have negative trust)
    score = max(0, score)

    return score


# STEP 2: Apply the function to every row of the DataFrame.
# axis=1 tells Pandas "apply this function row-by-row" (as opposed to
# axis=0, which would apply it column-by-column). The result is a new
# Series (a single column of values) that lines up with each row.
df["TrustScore"] = df.apply(calculate_trust_score, axis=1)

# STEP 3: Display the result so you can see how each trial scored.
# We'll show just the columns relevant to scoring, so it's easy to read.
print("--- Trust Scores for all 8 trials ---")
display_cols = ["NCTId", "BriefTitle", "EnrollmentCount", "HasResultsPosted", "TrustScore"]
print(df[display_cols])

# STEP 4 (optional sanity check): sort by TrustScore so you can quickly
# see your best- and worst-scoring trials at a glance.
print("\n--- Sorted by TrustScore (lowest first, i.e. biggest red flags) ---")
print(df[display_cols].sort_values("TrustScore"))

--- Trust Scores for all 8 trials ---
         NCTId                                         BriefTitle  \
0  NCT04720976  JAB-3312 Based Combination Therapy in Adult Pa...   
1  NCT04330664  Adagrasib in Combination With TNO155 in Patien...   
2  NCT06333678  A Study Comparing Sotorasib With Durvalumab in...   
3  NCT05054725  Combination Study of RMC-4630 and Sotorasib fo...   
4  NCT06314763            Rivaroxaban Sotorasib Interaction Study   
5  NCT04959981  A Study of Anti-Cancer Therapies Targeting the...   
6  NCT07143513  Real World Evaluation of Sotorasib Among Chine...   
7  NCT04933695  A Study of Sotorasib (AMG 510) in Participants...   

   EnrollmentCount  HasResultsPosted  TrustScore  
0               58             False          50  
1               86             False          50  
2               10             False          25  
3               47              True         100  
4               21             False          25  
5               24             Fal

In [ ]:
# ============================================================
# NORMALIZATION (v0) — Manual lookup table for drug codes -> generic names
# Builds on the existing 'df' DataFrame
# ============================================================

# STEP 1: Build our lookup dictionary.
# A Python dictionary maps "key" -> "value". Here: messy code -> clean name.
# IMPORTANT: This is a MANUAL, hand-typed map — it only knows about codes
# we've explicitly added. It is NOT the same as the spec's real normalization
# (PubChem/RxNorm API lookup + LLM extraction with confidence scores).
# Treat this as a temporary stand-in until that's built.
DRUG_CODE_MAP = {
    "MRTX849": "Adagrasib",
    "ADAGRASIB": "Adagrasib",
    "AMG 510": "Sotorasib",
    "AMG510": "Sotorasib",
    "SOTORASIB": "Sotorasib",
    "GDC-6036": "Divarasib",
    "DIVARASIB": "Divarasib",
    "TNO155": "TNO155",        # not yet normalized — no generic name assigned
    "DURVALUMAB": "Durvalumab" # already a generic name, included for consistency
}

# STEP 2: Write a function that takes one messy string (e.g. "MRTX849; TNO155")
# and returns a cleaned-up version using our lookup table.
def normalize_drug_string(raw_string):
    """
    Takes a semicolon-separated string of drug names/codes and returns
    a semicolon-separated string of normalized names, using DRUG_CODE_MAP.
    If a name isn't in our map, we keep it as-is (uppercased for consistency)
    rather than silently dropping it — we WANT to see what's still unmapped.
    """
    if pd.isna(raw_string) or raw_string == "":
        return ""

    # Split the messy string on "; " into a list of individual drug names
    # e.g. "MRTX849; TNO155" -> ["MRTX849", "TNO155"]
    individual_drugs = raw_string.split("; ")

    normalized_list = []
    for drug in individual_drugs:
        drug_clean = drug.strip()  # remove leading/trailing whitespace
        # .upper() makes matching case-insensitive: "amg510" and "AMG510"
        # both become "AMG510" before we look them up
        lookup_key = drug_clean.upper()

        if lookup_key in DRUG_CODE_MAP:
            normalized_list.append(DRUG_CODE_MAP[lookup_key])
        else:
            # Not in our map — keep the original so it doesn't disappear silently.
            # This is how we'll spot gaps in our lookup table later.
            normalized_list.append(drug_clean + " [UNMAPPED]")

    # Join the cleaned list back into a single string, removing duplicates
    # (e.g. if a trial lists "Adagrasib; ADAGRASIB" we don't want it twice)
    unique_normalized = list(dict.fromkeys(normalized_list))  # preserves order, drops dupes
    return "; ".join(unique_normalized)


# STEP 3: Apply this function to every row's Interventions column.
# .apply() here works on a single COLUMN (a Series), not the whole row,
# so each value gets passed into normalize_drug_string() one at a time.
df["Normalized_Drugs"] = df["Interventions"].apply(normalize_drug_string)

# STEP 4: Display the before/after so you can sanity-check the mapping
print("--- Before vs After Normalization ---")
print(df[["NCTId", "Interventions", "Normalized_Drugs"]])

# STEP 5: Flag any rows that still have unmapped codes, so you know
# exactly where your DRUG_CODE_MAP needs more entries
unmapped_rows = df[df["Normalized_Drugs"].str.contains(r"\[UNMAPPED\]", na=False)]
print(f"\n⚠️ {len(unmapped_rows)} row(s) contain unmapped drug codes:")
print(unmapped_rows[["NCTId", "Interventions", "Normalized_Drugs"]])

--- Before vs After Normalization ---
         NCTId                                      Interventions  \
0  NCT04720976  JAB-3312; Binimetinib; Pembrolizumab; Sotorasi...   
1  NCT04330664                                    MRTX849; TNO155   
2  NCT06333678                              Durvalumab; Sotorasib   
3  NCT05054725                                RMC-4630; Sotorasib   
4  NCT06314763                  Rivaroxaban 20mg; Sotorasib 960mg   
5  NCT04959981         ERAS-007; ERAS-601; Osimertinib; Sotorasib   
6  NCT07143513                                          Sotorasib   
7  NCT04933695                                          Sotorasib   

                                    Normalized_Drugs  
0  JAB-3312 [UNMAPPED]; Binimetinib [UNMAPPED]; P...  
1                                  Adagrasib; TNO155  
2                              Durvalumab; Sotorasib  
3                     RMC-4630 [UNMAPPED]; Sotorasib  
4  Rivaroxaban 20mg [UNMAPPED]; Sotorasib 960mg [...  
5  ERAS-00

In [ ]:
# ============================================================
# FINAL MERGE — Day 1 Master Output
# ============================================================

# STEP 1: Pick just the columns we want in the final deliverable,
# in the order we want them to appear.
final_cols = ["NCTId", "BriefTitle", "EnrollmentCount", "HasResultsPosted", "TrustScore", "Normalized_Drugs"]
df_final = df[final_cols].copy()  # .copy() avoids accidentally modifying the original df

# STEP 2: Save to CSV. index=False means don't write Pandas' internal
# row numbers (0,1,2...) as a column — we don't need them.
df_final.to_csv("pharma_evidence_v0_output.csv", index=False)

# STEP 3: Display it one last time so you can eyeball it before saving
print("--- Final Master Output (pharma_evidence_v0_output.csv) ---")
print(df_final)

print(f"\n✅ Saved {len(df_final)} rows to pharma_evidence_v0_output.csv")

--- Final Master Output (pharma_evidence_v0_output.csv) ---
         NCTId                                         BriefTitle  \
0  NCT04720976  JAB-3312 Based Combination Therapy in Adult Pa...   
1  NCT04330664  Adagrasib in Combination With TNO155 in Patien...   
2  NCT06333678  A Study Comparing Sotorasib With Durvalumab in...   
3  NCT05054725  Combination Study of RMC-4630 and Sotorasib fo...   
4  NCT06314763            Rivaroxaban Sotorasib Interaction Study   
5  NCT04959981  A Study of Anti-Cancer Therapies Targeting the...   
6  NCT07143513  Real World Evaluation of Sotorasib Among Chine...   
7  NCT04933695  A Study of Sotorasib (AMG 510) in Participants...   

   EnrollmentCount  HasResultsPosted  TrustScore  \
0               58             False          50   
1               86             False          50   
2               10             False          25   
3               47              True         100   
4               21             False          25   
5     

In [ ]:
# ============================================================
# PRODUCTION PIPELINE — Results Parsing + Power Calc + Knowledge Object Export
# ============================================================

import requests
import pandas as pd
import math
import json
from datetime import datetime

# ------------------------------------------------------------
# SECTION 0: GLOBAL CONFIG — clinical assumptions, isolated and labeled
# ------------------------------------------------------------
# These are NOT extracted from data. They are clinical judgment calls
# standing in until a pharmacologist reviews/replaces them.
# Source of defaults: typical second-line KRAS G12C NSCLC PFS literature
# (e.g. CodeBreaK/KRYSTAL-class trials) — NOT trial-specific, NOT derived
# from our own data, and NOT validated by a clinician on our team yet.
DEFAULT_TARGET_HAZARD_RATIO = 0.75   # HR we want the trial to be able to detect
ALPHA = 0.05                          # standard significance threshold (two-sided)
POWER = 0.80                          # standard target power (80%)
ASSUMED_EVENT_RATE = 0.85             # proportion of enrolled patients expected to have a PFS "event"
                                        # (progression/death) by analysis time — oncology PFS trials
                                        # typically see high event rates; this is also a placeholder
                                        # assumption, not derived from our data.

PHARMACOLOGIST_REVIEW_REQUIRED = True  # flips to False once a clinician signs off on the above

# ------------------------------------------------------------
# SECTION 1: Z-SCORES for alpha/power (standard statistical constants)
# ------------------------------------------------------------
from scipy.stats import norm

Z_ALPHA = norm.ppf(1 - ALPHA / 2)   # two-sided
Z_BETA = norm.ppf(POWER)

# ------------------------------------------------------------
# SECTION 2: POWER CALCULATION FUNCTION
# ------------------------------------------------------------
def calculate_required_n(hazard_ratio=DEFAULT_TARGET_HAZARD_RATIO,
                          event_rate=ASSUMED_EVENT_RATE,
                          z_alpha=Z_ALPHA, z_beta=Z_BETA):
    """
    Schoenfeld formula for required number of EVENTS to detect a given
    hazard ratio at the specified alpha/power (standard for time-to-event
    survival endpoints like PFS/OS in oncology trials).

    required_events = 4 * (z_alpha + z_beta)^2 / (ln(hazard_ratio))^2

    Then required_n = required_events / event_rate, since not every
    enrolled patient contributes an "event" (progression/death) by
    the time of analysis.
    """
    required_events = 4 * ((z_alpha + z_beta) ** 2) / (math.log(hazard_ratio) ** 2)
    required_n = required_events / event_rate
    return math.ceil(required_n)

REQUIRED_N = calculate_required_n()
print(f"Using HR={DEFAULT_TARGET_HAZARD_RATIO}, alpha={ALPHA}, power={POWER}")
print(f"--> Required enrollment per this assumption: {REQUIRED_N} patients\n")

def statistical_validity_score(row, required_n=REQUIRED_N):
    """
    Real Statistical Validity Score (0-100), built on:
    - actual EnrollmentCount vs. calculated required_n (power check)
    - actual extracted p-value (if available)
    - actual extracted confidence interval presence
    - HasResultsPosted

    NOTE: the 'required_n' threshold itself rests on DEFAULT_TARGET_HAZARD_RATIO,
    a clinical assumption flagged above as pending pharmacologist review.
    """
    score = 100
    notes = []

    # --- Power check (now using a real formula, not n < 30) ---
    enrollment = row.get("EnrollmentCount")
    if pd.isna(enrollment):
        score -= 25
        notes.append("Enrollment count missing — cannot verify power")
    elif enrollment < required_n:
        score -= 25
        notes.append(f"Underpowered: enrolled {enrollment} vs. required {required_n} "
                      f"(assumes HR={DEFAULT_TARGET_HAZARD_RATIO})")
    else:
        notes.append(f"Adequately powered: enrolled {enrollment} vs. required {required_n} "
                      f"(assumes HR={DEFAULT_TARGET_HAZARD_RATIO})")

    # --- P-value suspicion check (real, on extracted p-value) ---
    p_val = row.get("PrimaryEndpoint_PValue")
    if p_val is not None and not pd.isna(p_val):
        if 0.04 < p_val <= 0.05:
            score -= 10
            notes.append(f"Borderline p-value ({p_val}) — elevated p-hacking risk")
    else:
        notes.append("No p-value extracted — cannot assess")

    # --- CI reporting check ---
    ci_lower = row.get("PrimaryEndpoint_CI_Lower")
    if ci_lower is None or pd.isna(ci_lower):
        score -= 15
        notes.append("No confidence interval reported")

    # --- Missing results check ---
    if row.get("HasResultsPosted") == False:
        score -= 50
        notes.append("No results posted — cannot verify any claims")

    score = max(0, score)
    return score, notes


# ------------------------------------------------------------
# SECTION 3: RESULTS SECTION PARSER
# ------------------------------------------------------------
def fetch_full_study(nct_id):
    """Fetch the FULL study record (including resultsSection) for one NCT ID."""
    url = f"https://clinicaltrials.gov/api/v2/studies/{nct_id}"
    response = requests.get(url, params={"format": "json"})
    if response.status_code == 200:
        return response.json()
    else:
        print(f"⚠️ Failed to fetch {nct_id}: status {response.status_code}")
        return None


def parse_results_section(study_json):
    """
    Digs into resultsSection.outcomeMeasuresModule to find the PRIMARY
    outcome's measured values, p-value, and confidence interval.
    Returns a dict of extracted values, with None for anything not found
    (real null, not a guess) and a confidence flag, consistent with the
    spec's own "confidence: high/medium/low/not_extracted" pattern.
    """
    extracted = {
        "PrimaryEndpoint_Description": None,
        "PrimaryEndpoint_InterventionValue": None,
        "PrimaryEndpoint_ComparatorValue": None,
        "PrimaryEndpoint_Unit": None,
        "PrimaryEndpoint_PValue": None,
        "PrimaryEndpoint_CI_Lower": None,
        "PrimaryEndpoint_CI_Upper": None,
        "extraction_confidence": "not_extracted"
    }

    results_section = study_json.get("resultsSection")
    if not results_section:
        return extracted  # no results posted — stays all-None, honestly

    outcome_module = results_section.get("outcomeMeasuresModule", {})
    outcome_measures = outcome_module.get("outcomeMeasures", [])

    # Find the first outcome flagged as PRIMARY
    primary_outcomes = [om for om in outcome_measures if om.get("type") == "PRIMARY"]
    if not primary_outcomes:
        return extracted

    primary = primary_outcomes[0]
    extracted["PrimaryEndpoint_Description"] = primary.get("title")
    extracted["PrimaryEndpoint_Unit"] = primary.get("unitOfMeasure")

    # Outcome measure groups/classes hold the actual numbers per arm
    classes = primary.get("classes", [])
    if classes:
        categories = classes[0].get("categories", [])
        if categories:
            measurements = categories[0].get("measurements", [])
            # measurements is a list, one entry per study arm (e.g. drug vs. comparator)
            if len(measurements) >= 1:
                extracted["PrimaryEndpoint_InterventionValue"] = measurements[0].get("value")
            if len(measurements) >= 2:
                extracted["PrimaryEndpoint_ComparatorValue"] = measurements[1].get("value")

    # Statistical analyses (p-value, CI) live in a separate block
    analyses = primary.get("analyses", [])
    if analyses:
        analysis = analyses[0]
        p_val_raw = analysis.get("pValue")
        try:
            extracted["PrimaryEndpoint_PValue"] = float(p_val_raw) if p_val_raw else None
        except (ValueError, TypeError):
            extracted["PrimaryEndpoint_PValue"] = None  # e.g. "<0.001" won't cast cleanly

        ci_lower = analysis.get("ciLowerLimit")
        ci_upper = analysis.get("ciUpperLimit")
        try:
            extracted["PrimaryEndpoint_CI_Lower"] = float(ci_lower) if ci_lower else None
            extracted["PrimaryEndpoint_CI_Upper"] = float(ci_upper) if ci_upper else None
        except (ValueError, TypeError):
            pass

    # Mark confidence based on what we actually got
    if extracted["PrimaryEndpoint_PValue"] is not None:
        extracted["extraction_confidence"] = "high"
    elif extracted["PrimaryEndpoint_Description"] is not None:
        extracted["extraction_confidence"] = "medium"
    else:
        extracted["extraction_confidence"] = "low"

    return extracted


# ------------------------------------------------------------
# SECTION 4: RUN THE PARSER ACROSS OUR 8 TRIALS
# ------------------------------------------------------------
results_rows = []
for nct_id in df["NCTId"]:
    print(f"Fetching full record for {nct_id}...")
    study_json = fetch_full_study(nct_id)
    if study_json:
        parsed = parse_results_section(study_json)
    else:
        parsed = parse_results_section({})  # all-None fallback
    parsed["NCTId"] = nct_id
    results_rows.append(parsed)

df_results = pd.DataFrame(results_rows)

# Merge the newly-parsed results columns onto our existing df
df = df.merge(df_results, on="NCTId", how="left")

print("\n--- Parsed results section data ---")
print(df[["NCTId", "PrimaryEndpoint_Description", "PrimaryEndpoint_PValue",
          "PrimaryEndpoint_CI_Lower", "PrimaryEndpoint_CI_Upper",
          "extraction_confidence"]])

# ------------------------------------------------------------
# SECTION 5: APPLY THE REAL TRUST SCORE
# ------------------------------------------------------------
score_results = df.apply(statistical_validity_score, axis=1)
df["TrustScore"] = score_results.apply(lambda x: x[0])
df["TrustScore_Notes"] = score_results.apply(lambda x: x[1])

print("\n--- Final Trust Scores (with reasoning notes) ---")
for _, row in df.iterrows():
    print(f"\n{row['NCTId']} — Score: {row['TrustScore']}")
    for note in row["TrustScore_Notes"]:
        print(f"   - {note}")

# ------------------------------------------------------------
# SECTION 6: BUILD KNOWLEDGE OBJECTS — only with what we actually have
# ------------------------------------------------------------
def build_knowledge_object(row, index):
    ko = {
        "ko_id": f"KO-2026-NSCLC-{str(index).zfill(5)}",
        "version": "0.1",  # honestly versioned — this is a first pass, not v2.1 like the screenshot example
        "created_at": datetime.utcnow().isoformat() + "Z",
        "status": "draft_pending_pharmacologist_review",
        "source": {
            "primary_source": "clinicaltrials_gov",
            "nct_id": row["NCTId"]
        },
        "study_metadata": {
            "title": row["BriefTitle"],
            "phase": row["Phase"],
            "enrollment_actual": None if pd.isna(row["EnrollmentCount"]) else int(row["EnrollmentCount"]),
            "lead_sponsor": row["LeadSponsor"]
        },
        "entities": {
            "intervention": {
                "normalized_drugs": row["Normalized_Drugs"],
                # NOTE: no rxnorm_id/chembl_id/smiles — that requires real PubChem/RxNorm
                # API calls we have not built yet. Intentionally omitted, not guessed.
                "confidence": "medium" if "[UNMAPPED]" not in row["Normalized_Drugs"] else "low"
            }
            # NOTE: no "target" or full "indication" block — we never extracted ICD-11/MeSH
            # codes. Omitted rather than fabricated.
        },
        "pico": {
            "population": {
                "sample_size": None if pd.isna(row["EnrollmentCount"]) else int(row["EnrollmentCount"]),
                "description": None,  # not extracted — would require eligibilityModule parsing (not built yet)
                "confidence": "not_extracted"
            },
            "outcomes": [
                {
                    "type": "primary",
                    "description": row.get("PrimaryEndpoint_Description"),
                    "confidence": row.get("extraction_confidence")
                }
            ]
        },
        "results": {
            "primary_endpoint": {
                "outcome": row.get("PrimaryEndpoint_Description"),
                "intervention_value": row.get("PrimaryEndpoint_InterventionValue"),
                "comparator_value": row.get("PrimaryEndpoint_ComparatorValue"),
                "unit": row.get("PrimaryEndpoint_Unit"),
                "p_value": row.get("PrimaryEndpoint_PValue"),
                "confidence_interval_95": [
                    row.get("PrimaryEndpoint_CI_Lower"),
                    row.get("PrimaryEndpoint_CI_Upper")
                ] if row.get("PrimaryEndpoint_CI_Lower") is not None else None
            },
            "results_posted": bool(row["HasResultsPosted"])
        },
        "trust_signals": {
            "statistical_validity_score": int(row["TrustScore"]),
            "statistical_validity_notes": row["TrustScore_Notes"],
            "statistical_validity_assumptions": {
                "target_hazard_ratio": DEFAULT_TARGET_HAZARD_RATIO,
                "assumed_event_rate": ASSUMED_EVENT_RATE,
                "alpha": ALPHA,
                "power": POWER,
                "pharmacologist_reviewed": not PHARMACOLOGIST_REVIEW_REQUIRED
            },
            # NOT calculated tonight — each requires data/sources we haven't built:
            "replication_confidence_score": None,   # needs cross-trial matching logic
            "coi_flags": None,                        # needs OpenPayments + sponsor-type logic
            "retraction_status": None                 # needs Retraction Watch cross-reference
        },
        "agent_rationale": None  # NOT generated — writing this requires synthesizing real
                                   # clinical judgment from complete PICO+results+trust data,
                                   # which we don't have yet. A templated paragraph here would
                                   # be fabricated narrative wearing a real-looking field name.
    }
    return ko


knowledge_objects = [build_knowledge_object(row, i+1) for i, row in df.iterrows()]

with open("kras_g12c_knowledge_objects.json", "w") as f:
    json.dump(knowledge_objects, f, indent=2, default=str)

print(f"\n✅ Saved {len(knowledge_objects)} Knowledge Objects to kras_g12c_knowledge_objects.json")
print("⚠️  status='draft_pending_pharmacologist_review' on every object — by design.")

Using HR=0.75, alpha=0.05, power=0.8
--> Required enrollment per this assumption: 447 patients

Fetching full record for NCT04720976...
Fetching full record for NCT04330664...
Fetching full record for NCT06333678...
Fetching full record for NCT05054725...
Fetching full record for NCT06314763...
Fetching full record for NCT04959981...
Fetching full record for NCT07143513...
Fetching full record for NCT04933695...

--- Parsed results section data ---
         NCTId                        PrimaryEndpoint_Description  \
0  NCT04720976                                               None   
1  NCT04330664                                               None   
2  NCT06333678                                               None   
3  NCT05054725  Objective Response Rate (ORR) as Assessed Per ...   
4  NCT06314763                                               None   
5  NCT04959981                                               None   
6  NCT07143513                                               Non

/tmp/ipykernel_2192/3990653706.py:247: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",
